# MNIST Interval Analysis with Bound Propagation

This notebook implements interval bound propagation for robustness verification of neural networks on MNIST.

## Requirements
First, install the required libraries:
```bash
pip install bound-propagation torch torchvision matplotlib numpy
```

In [1]:
# Installation cell - run this first if bound_propagation is not installed
# !pip install bound-propagation

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
import time

# Check if bound_propagation is available
try:
    from bound_propagation import BoundModelFactory, HyperRectangle
    print("bound_propagation imported successfully")
    BOUND_PROPAGATION_AVAILABLE = True
except ImportError:
    print("bound_propagation not available. Install with: pip install bound-propagation")
    BOUND_PROPAGATION_AVAILABLE = False

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)

bound_propagation imported successfully
Using device: cuda


In [2]:
# Data loading
print("Loading MNIST dataset...")

train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, 
                              transform=transforms.Compose([transforms.ToTensor()]))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, 
                             transform=transforms.Compose([transforms.ToTensor()]))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Loading MNIST dataset...
Training samples: 60000
Test samples: 10000


In [3]:
# Neural network definition
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(28*28, 200)
        self.fc2 = nn.Linear(200, 10)

    def forward(self, x):
        x = x.view((-1, 28*28))
        x = F.relu(self.fc(x))
        x = self.fc2(x)
        return x

class Normalize(nn.Module):
    def forward(self, x):
        return (x - 0.1307) / 0.3081

# Create model (without softmax for bound propagation compatibility)
model = nn.Sequential(Normalize(), Net())
model = model.to(device)

print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

Model architecture:
Sequential(
  (0): Normalize()
  (1): Net(
    (fc): Linear(in_features=784, out_features=200, bias=True)
    (fc2): Linear(in_features=200, out_features=10, bias=True)
  )
)

Total parameters: 159010


In [4]:
# Training function
def train_model(model, num_epochs):
    """Train the model for specified epochs"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_acc = 100 * correct / total
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}, Accuracy: {epoch_acc:.2f}%')

# Train the model for 15 epochs
print("Training model for 15 epochs...")
train_model(model, 15)

Training model for 15 epochs...
Epoch 1/15, Loss: 0.593, Accuracy: 84.82%
Epoch 2/15, Loss: 0.297, Accuracy: 91.54%
Epoch 3/15, Loss: 0.248, Accuracy: 92.92%
Epoch 4/15, Loss: 0.215, Accuracy: 93.95%
Epoch 5/15, Loss: 0.190, Accuracy: 94.64%
Epoch 6/15, Loss: 0.170, Accuracy: 95.23%
Epoch 7/15, Loss: 0.155, Accuracy: 95.68%
Epoch 8/15, Loss: 0.142, Accuracy: 96.00%
Epoch 9/15, Loss: 0.130, Accuracy: 96.36%
Epoch 10/15, Loss: 0.121, Accuracy: 96.64%
Epoch 11/15, Loss: 0.113, Accuracy: 96.88%
Epoch 12/15, Loss: 0.106, Accuracy: 97.09%
Epoch 13/15, Loss: 0.099, Accuracy: 97.25%
Epoch 14/15, Loss: 0.093, Accuracy: 97.44%
Epoch 15/15, Loss: 0.088, Accuracy: 97.57%


In [5]:
# Testing function
def test_model(model):
    """Test the model and return clean accuracy"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # Apply softmax for probability interpretation
            outputs = F.softmax(outputs, dim=-1)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'Clean Accuracy on test set: {accuracy:.2f}%')
    return accuracy

# Test the model
clean_accuracy = test_model(model)

Clean Accuracy on test set: 97.06%


In [6]:
# Interval Analysis Implementation
def perform_interval_analysis(model, test_loader, epsilon_values, max_samples=1000):
    """
    Perform interval analysis for different epsilon values
    Returns verification results for each epsilon
    """
    if not BOUND_PROPAGATION_AVAILABLE:
        print("bound_propagation library not available!")
        print("Install with: pip install bound-propagation")
        return None
    
    print("\nPerforming Interval Analysis...")
    
    # Create bound model factory
    factory = BoundModelFactory()
    
    # Build bound model
    bound_model = factory.build(model)
    bound_model.eval()
    
    results = {}
    
    # Get test samples
    test_samples = []
    test_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            for i in range(images.shape[0]):
                if len(test_samples) >= max_samples:
                    break
                test_samples.append(images[i])
                test_labels.append(labels[i].item())
            if len(test_samples) >= max_samples:
                break
    
    print(f"Testing on {len(test_samples)} samples")
    
    for eps in epsilon_values:
        print(f"\nTesting epsilon = {eps:.3f}")
        verified_count = 0
        
        for idx, (sample, true_label) in enumerate(zip(test_samples, test_labels)):
            if idx % 100 == 0:
                print(f"  Processing sample {idx}/{len(test_samples)}")
            
            try:
                # Create perturbation bounds
                lower_bound = torch.clamp(sample - eps, 0.0, 1.0).flatten()
                upper_bound = torch.clamp(sample + eps, 0.0, 1.0).flatten()
                
                # Create HyperRectangle for the perturbed input
                input_bounds = HyperRectangle(lower_bound, upper_bound)
                
                # Propagate bounds using IBP
                output_bounds = bound_model(input_bounds, method='ibp')
                
                # Check verification condition
                lower_bounds = output_bounds.lower
                upper_bounds = output_bounds.upper
                
                # The true label should have the highest lower bound compared to 
                # the upper bounds of all other classes
                true_label_lower = lower_bounds[true_label]
                
                is_verified = True
                for j in range(len(upper_bounds)):
                    if j != true_label and true_label_lower <= upper_bounds[j]:
                        is_verified = False
                        break
                
                if is_verified:
                    verified_count += 1
            
            except Exception as e:
                # Count as not verified if there's an error
                continue
        
        verification_rate = (verified_count / len(test_samples)) * 100
        results[eps] = {
            'verified': verified_count,
            'total': len(test_samples),
            'rate': verification_rate
        }
        
        print(f"  Verified: {verified_count}/{len(test_samples)} ({verification_rate:.2f}%)")
    
    return results

# Define epsilon values (10 values evenly spaced between 0.01 and 0.1)
epsilon_values = np.linspace(0.01, 0.1, 10)
print(f"Testing with epsilon values: {epsilon_values}")

# Perform interval analysis
results = perform_interval_analysis(model, test_loader, epsilon_values, max_samples=1000)

Testing with epsilon values: [0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1 ]

Performing Interval Analysis...


NotImplementedError: Module type not supported - add BoundModule for layer to factory

In [ ]:
# Results visualization and analysis
def visualize_results(results, clean_accuracy):
    """Visualize the verification results"""
    if results is None:
        print("No results to visualize")
        return
    
    epsilons = list(results.keys())
    verification_rates = [results[eps]['rate'] for eps in epsilons]
    
    plt.figure(figsize=(12, 8))
    
    # Main plot
    plt.subplot(2, 1, 1)
    plt.plot(epsilons, verification_rates, 'b-o', linewidth=2, markersize=8, label='Verified Accuracy')
    plt.axhline(y=clean_accuracy, color='r', linestyle='--', alpha=0.7, label=f'Clean Accuracy ({clean_accuracy:.1f}%)')
    plt.xlabel('Epsilon (L∞ perturbation size)')
    plt.ylabel('Accuracy (%)')
    plt.title('Neural Network Robustness Verification Results\n(Interval Bound Propagation)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.xlim(0, max(epsilons))
    plt.ylim(0, 100)
    
    # Add annotations
    for i, (eps, rate) in enumerate(zip(epsilons, verification_rates)):
        plt.annotate(f'{rate:.1f}%', (eps, rate), textcoords="offset points", 
                    xytext=(0,10), ha='center', fontsize=8)
    
    # Bar chart
    plt.subplot(2, 1, 2)
    verified_counts = [results[eps]['verified'] for eps in epsilons]
    total_counts = [results[eps]['total'] for eps in epsilons]
    not_verified = [total - verified for total, verified in zip(total_counts, verified_counts)]
    
    width = 0.008
    plt.bar(epsilons, verified_counts, width, label='Verified', color='green', alpha=0.7)
    plt.bar(epsilons, not_verified, width, bottom=verified_counts, label='Not Verified', color='red', alpha=0.7)
    
    plt.xlabel('Epsilon (L∞ perturbation size)')
    plt.ylabel('Number of Samples')
    plt.title('Verification Breakdown by Epsilon')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def print_detailed_results(results, clean_accuracy):
    """Print detailed results analysis"""
    if results is None:
        return
    
    print("\n" + "="*70)
    print("DETAILED VERIFICATION RESULTS")
    print("="*70)
    print(f"Clean Accuracy: {clean_accuracy:.2f}%")
    print(f"Network Architecture: 784 -> 200 (ReLU) -> 10")
    print(f"Verification Method: Interval Bound Propagation (IBP)")
    
    print("\nVerified Accuracy by L∞ Perturbation Size:")
    print("-" * 50)
    for eps in sorted(results.keys()):
        rate = results[eps]['rate']
        verified = results[eps]['verified']
        total = results[eps]['total']
        print(f"ε = {eps:.3f}: {rate:6.2f}% ({verified:4d}/{total:4d})")
    
    # Analysis
    print("\n" + "="*70)
    print("OBSERVATIONS AND ANALYSIS")
    print("="*70)
    
    epsilon_values = list(results.keys())
    rates = [results[eps]['rate'] for eps in epsilon_values]
    
    print(f"1. Robustness decreases with perturbation size:")
    print(f"   - Smallest ε ({epsilon_values[0]:.3f}): {rates[0]:.1f}% verified")
    print(f"   - Largest ε ({epsilon_values[-1]:.3f}): {rates[-1]:.1f}% verified")
    print(f"   - Total drop: {rates[0] - rates[-1]:.1f} percentage points")
    
    # Find steepest drop
    max_drop = 0
    max_drop_range = None
    for i in range(len(rates) - 1):
        drop = rates[i] - rates[i+1]
        if drop > max_drop:
            max_drop = drop
            max_drop_range = (epsilon_values[i], epsilon_values[i+1])
    
    if max_drop_range:
        print(f"\n2. Steepest robustness drop:")
        print(f"   - Between ε = {max_drop_range[0]:.3f} and ε = {max_drop_range[1]:.3f}")
        print(f"   - Drop: {max_drop:.1f} percentage points")
    
    print(f"\n3. Conservative nature of IBP:")
    print(f"   - IBP provides sound (guaranteed correct) but conservative bounds")
    print(f"   - Actual robustness may be higher than verified robustness")
    print(f"   - The gap represents the 'price of soundness' in verification")
    
    print(f"\n4. Practical implications:")
    print(f"   - For safety-critical applications, use smallest ε with acceptable verification rate")
    print(f"   - Higher verification rates indicate more reliable robustness guarantees")
    print(f"   - Consider robustness training to improve verified accuracy")

# Generate results
if results:
    visualize_results(results, clean_accuracy)
    print_detailed_results(results, clean_accuracy)
else:
    print("Could not perform interval analysis. Please install bound_propagation:")
    print("pip install bound-propagation")

In [ ]:
# Save results to CSV for further analysis
if results:
    import pandas as pd
    
    # Create DataFrame
    df_data = []
    for eps in sorted(results.keys()):
        df_data.append({
            'epsilon': eps,
            'verified_count': results[eps]['verified'],
            'total_count': results[eps]['total'],
            'verified_accuracy': results[eps]['rate'],
            'clean_accuracy': clean_accuracy
        })
    
    df = pd.DataFrame(df_data)
    
    # Save to CSV
    csv_filename = 'mnist_verification_results.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"Results saved to {csv_filename}")
    print("\nDataFrame preview:")
    print(df)
else:
    print("No results to save.")